In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

import joblib
import os


In [2]:
df = pd.read_csv(r"C:\Users\A I TECH\Project\SentiCare-Capstone\data\raw\Stress_Dataset.csv")
print(df.shape)
df.head()


(843, 26)


,Gender,Age,Have you recently experienced stress in your life?,Have you noticed a rapid heartbeat or palpitations?,Have you been dealing with anxiety or tension recently?,Do you face any sleep problems or difficulties falling asleep?,Have you been dealing with anxiety or tension recently?.1,Have you been getting headaches more often than usual?,Do you get irritated easily?,Do you have trouble concentrating on your academic tasks?,...,Are you facing any difficulties with your professors or instructors?,Is your working environment unpleasant or stressful?,Do you struggle to find time for relaxation and leisure activities?,Is your hostel or home environment causing you difficulties?,Do you lack confidence in your academic performance?,Do you lack confidence in your choice of academic subjects?,Academic and extracurricular activities conflicting for you?,Do you attend classes regularly?,Have you gained/lost weight?,Which type of stress do you primarily experience?
0,0,20,3,4,2,5,1,2,1,2,...,3,1,4,1,2,1,3,1,2,Eustress (Positive Stress) - Stress that motiv...
1,0,20,2,3,2,1,1,1,1,4,...,3,2,1,1,3,2,1,4,2,Eustress (Positive Stress) - Stress that motiv...
2,0,20,5,4,2,2,1,3,4,2,...,2,2,2,1,4,1,1,2,1,Eustress (Positive Stress) - Stress that motiv...
3,1,20,3,4,3,2,2,3,4,3,...,1,1,2,1,2,1,1,5,3,Eustress (Positive Stress) - Stress that motiv...
4,0,20,3,3,3,2,2,4,4,4,...,2,3,1,2,2,4,2,2,2,Eustress (Positive Stress) - Stress that motiv...


In [3]:
TARGET_COL = "Which type of stress do you primarily experience?"

X = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL]

print(X.shape, y.shape)


(843, 25) (843,)


# Encode target (classification requirement)

In [4]:
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

label_encoder.classes_


array(['Distress (Negative Stress) - Stress that causes anxiety and impairs well-being.',
       'Eustress (Positive Stress) - Stress that motivates and enhances performance.',
       'No Stress - Currently experiencing minimal to no stress.'],
      dtype=object)

0 = Distress
1 = Eustress
2 = No Stress


In [5]:
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)


In [6]:
categorical_cols = [
    col for col in X.columns if X[col].dtype == "object"
]

numerical_cols = [
    col for col in X.columns if X[col].dtype in ["int64", "float64"]
]


Build preprocessing pipeline

In [7]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numerical_cols),
    ("cat", categorical_pipeline, categorical_cols)
])


In [8]:
model = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    class_weight="balanced"
)



In [9]:
full_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", model)
])


In [10]:
full_pipeline.fit(X_train, y_train)


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median'))]),
                                                  ['Gender', 'Age',
                                                   'Have you recently '
                                                   'experienced stress in your '
                                                   'life?',
                                                   'Have you noticed a rapid '
                                                   'heartbeat or palpitations?',
                                                   'Have you been dealing with '
                                                   'anxiety or tension '
                                                   'recently?',
                                                   'Do you face any sleep '
                                                   'problems or difficulties '
                                                   'fal...
                                                   'Academic and '
                                                   'extracurricular activities '
                                                   'conflicting for you?',
                                                   'Do you attend classes '
                                                   'regularly?',
                                                   'Have you gained/lost '
                                                   'weight?']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  [])])),
                ('model',
                 RandomForestClassifier(class_weight='balanced',
                                        n_estimators=300, random_state=42))])

In [11]:
val_preds = full_pipeline.predict(X_val)

print("Validation Accuracy:", accuracy_score(y_val, val_preds))
print("\nClassification Report:")
print(classification_report(
    y_val,
    val_preds,
    target_names=label_encoder.classes_
))


Validation Accuracy: 0.9408284023668639

Classification Report:
                                                                                 precision    recall  f1-score   support

Distress (Negative Stress) - Stress that causes anxiety and impairs well-being.       1.00      0.50      0.67         6
   Eustress (Positive Stress) - Stress that motivates and enhances performance.       0.94      1.00      0.97       154
                       No Stress - Currently experiencing minimal to no stress.       1.00      0.22      0.36         9

                                                                       accuracy                           0.94       169
                                                                      macro avg       0.98      0.57      0.67       169
                                                                   weighted avg       0.94      0.94      0.93       169



In [12]:
from pathlib import Path

ARTIFACT_DIR = Path(
    r"C:\Users\A I TECH\Project\SentiCare-Capstone\artifacts\stress_classification"
)

ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)




In [13]:
joblib.dump(
    {
        "pipeline": full_pipeline,
        "label_encoder": label_encoder,
        "feature_columns": X.columns.tolist()
    },
    ARTIFACT_DIR / "stress_bundle.joblib"
)

print("\nSaved stress model bundle to:", ARTIFACT_DIR)


Saved stress model bundle to: C:\Users\A I TECH\Project\SentiCare-Capstone\artifacts\stress_classification
